# Portfolio construction 3 of 3: how to get there

Notebooks: [1 hold](portfolio_construction_1_hold.ipynb) · [2 scale](portfolio_construction_2_scale.ipynb) · [3 trade](portfolio_construction_3_trade.ipynb), matching `src/portfolio_construction/{hold,scale,trade}`. In-sample, 1976-2014, DCC-GARCH risk model with the rescale and HRP-within-class hold sizing (the current defaults), primary cost 2 bps per traded notional (`primary_cost_bps`); data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`).

## Goal

Improve the model. Here: cut turnover and cost without hurting returns, and check that the band earns its place.

## Current pipeline

**What we are doing.** Move from last month's positions to the target without trading small differences, then make sure the limits still hold after trading costs. This is the only stage with memory of past positions.

**Inputs**
- Previous executed positions and the target book.
- Band width 10% and the cost model (2 bps per traded notional).
- Covariance `Q` and the gross and risk limits.

**How we process them** (`trade/`)
1. **Band** (`bands.py`), market by market. If the new target has the same sign as the current position and the position is within ±10% of the target, keep it. If it is outside, trade only to the nearest edge of the band. Entries, exits and sign flips trade straight to the target.
2. **Executed limits** (`feasibility.py`). Price the trades. If the book plus fees would break the gross cap or risk ceiling, cut all positions by one factor, found by bisection.

*Example:* a target of 2.0% with 1.9% held: no trade. With 1.5% held: buy to 1.8%. With -0.5% held (a sign flip): go to 2.0%.

**Output.** The executed positions, which the ledger charges with costs.

## Implications

- This is the only stateful stage and the only one that uses the cost model.
- Costs do not set the band width. They enter at step 7 (fees reduce room under the limits) and in the ledger.
- Held positions sit off target by up to 10%, and step 7 can override the band.

## Results and interpretation

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]


from risk_model_study import load_study, paired_sharpe_difference, run, summarize
from run_pc_scale import model_parts

pd.options.display.float_format = "{:.3f}".format
study = load_study()
parts = model_parts(study, "configured")
ledgers = run(study, parts.risk_model)
print(f"trend ledger months: {len(ledgers['trend'])}")

trend ledger months: 468


In [2]:
trend_ledger = ledgers["trend"]
pd.Series(
    {
        "months executed limit override (after fees)": trend_ledger.limit_override.mean(),
        "annual turnover": summarize(trend_ledger)["turnover_ann"],
        "mean annual cost drag": summarize(trend_ledger)["mean_cost_ann"],
    }
).to_frame("trend")

,trend
months executed limit override (after fees),0.201
annual turnover,13.931
mean annual cost drag,0.003


- **Executed limits override the band in about 20% of months:** the post-fee cut, not the band, sets the position.
- **Turnover is 13.9x a year, cost drag about 0.28% a year** at 2 bps per traded notional (HRP sizing; inverse-volatility sizing gave 7.4x). This is the reference any change must beat.
- The band-off result is in T1 below.

## AFML improvements

Read: Ch. 10 (10.4-10.5), Ch. 14 (14.7.3).

| Idea | What the book says | Use here |
|---|---|---|
| Averaging and discretising bets (10.4-10.5) | Averaging sizes over active bets and rounding to steps `d` cut turnover from small resizes. | The ±10% band does the same job. |
| Deflated Sharpe (14.7.3) | Judge a Sharpe against the maximum expected from the number of trials. | Each band or discretisation variant is a trial. |

## Suggested changes and experiments

| # | Experiment | Metrics |
|---|---|---|
| T1 | Band sweep: 0.0 (off) against 0.1, plus wider bands (`bands` in `risk_config.json` already lists 0.0 and 0.1) | Turnover, cost, Sharpe, tracking risk to target (`forecast_tracking_risk`) |
| P4 | Size discretisation in steps of `d` against the band at the same turnover. Expect little gain; skip if P1-P3 in notebook 1 show none. | Same, at matched turnover |
| T2 | Let cost set the band width (wider where trades are expensive) [mine] | Turnover, cost, Sharpe |

## Tests

Logic is in this notebook (no script, so the sealed source snapshot is untouched). Every run uses the configured DCC-GARCH model with the rescale and the current HRP-within-class hold sizing: the same limited TREND targets, the executed-limit step and the same ledger, changing only the trade rule. Reference: the 10% band at 2 bps.

**Assumptions I took.**
1. *Band widths:* 0 (off), 5, 10, 15, 20 and 30%, at 2 bps and at 5 bps, because turnover is high and the cost level decides what a band is worth.
2. *Discretisation (P4)* rounds each target to a multiple `d` of NAV (0.25, 0.5, 1 and 2%) and trades to it with no band, then applies the same executed limits. The tracking risk it reports is to the rounded target.
3. *T2 (cost sets the band width) is not run:* the cost model charges one rate per unit of traded notional in every market (`LinearBpsCost`), so there is no cost difference between markets to size a band on. It needs per-market costs, which the repo does not have.
4. *Trial count:* 6 bands plus 4 discretisation steps, 9 differences from the reference at 2 bps; 25 and 100 are sensitivities.

In [3]:
import numpy as np
from src.backtest.engine import simulate
from src.backtest.qps_baseline import forecasts_from
from src.portfolio_construction import BandPolicy
from src.transaction_cost_model import LinearBpsCost

settings, base, returns, source, known, meta = study.inputs
targets, covariance, _ = forecasts_from(parts, returns, meta, base)
trend = targets["trend"]
sample = returns.loc[base["start"] : base["end"]]


class DiscretePolicy:
    """Round each target to a multiple of `step`, trade to it with no band, then apply the limits."""

    def __init__(self, step):
        self.step = step
        self.inner = BandPolicy(covariance, parts.limits, 0.0)

    def __call__(self, month, previous, target, *args, **kwargs):
        return self.inner(month, previous, np.round(target / self.step) * self.step, *args, **kwargs)


def run_trade(policy, bps):
    ledger, _ = simulate(
        sample, trend, LinearBpsCost(bps), source.loc[sample.index],
        position_policy=policy, source_known_at=known.loc[sample.index],
    )
    return ledger


def row(ledger):
    stats = summarize(ledger)
    return {
        "sharpe": stats["sharpe"], "vol_ann": stats["vol_ann"], "turnover_ann": stats["turnover_ann"],
        "cost_ann": stats["mean_cost_ann"], "max_drawdown": stats["max_drawdown"],
        "tracking_risk": ledger.forecast_tracking_risk.mean(),
        "limit_override": ledger.limit_override.mean(),
    }

### T1: band sweep

**What we are doing.** Test whether the 10% band earns its place against no band and wider bands, at 2 bps and at 5 bps.

**Inputs.** The same TREND targets, covariance and limits; only the band width changes.

**Output.** Sharpe, turnover, cost, drawdown, mean forecast tracking risk to the target, and the share of months the executed limits override the band.

In [4]:
BANDS = (0.0, 0.05, 0.10, 0.15, 0.20, 0.30)
band_runs = {
    (bps, band): run_trade(BandPolicy(covariance, parts.limits, band), bps)
    for bps in (2, 5)
    for band in BANDS
}
t1 = pd.DataFrame({k: row(v) for k, v in band_runs.items()}).T
t1.index.names = ["bps", "band"]
t1

sharpe  vol_ann  turnover_ann  cost_ann  max_drawdown  \
bps band                                                           
2   0.000   1.329    0.086        16.562     0.003        -0.107   
    0.050   1.325    0.085        15.132     0.003        -0.106   
    0.100   1.323    0.084        13.931     0.003        -0.106   
    0.150   1.319    0.084        12.923     0.003        -0.102   
    0.200   1.314    0.083        12.060     0.002        -0.098   
    0.300   1.310    0.081        10.673     0.002        -0.112   
5   0.000   1.271    0.086        16.558     0.008        -0.111   
    0.050   1.272    0.085        15.134     0.008        -0.109   
    0.100   1.274    0.084        13.934     0.007        -0.109   
    0.150   1.273    0.084        12.925     0.006        -0.105   
    0.200   1.271    0.083        12.063     0.006        -0.100   
    0.300   1.272    0.081        10.676     0.005        -0.112   

           tracking_risk  limit_override  
bps band                                  
2   0.000          0.000           0.981  
    0.050          0.003           0.267  
    0.100          0.006           0.201  
    0.150          0.009           0.145  
    0.200          0.012           0.132  
    0.300          0.017           0.092  
5   0.000          0.000           0.981  
    0.050          0.003           0.274  
    0.100          0.006           0.205  
    0.150          0.009           0.152  
    0.200          0.012           0.132  
    0.300          0.017           0.092

### P4: size discretisation against the band

**What we are doing.** Test whether rounding positions to steps of `d` cuts turnover as the band does, at the same cost level.

**Output.** The same table, with the 10% band as the comparison.

In [5]:
STEPS = (0.0025, 0.005, 0.01, 0.02)
step_runs = {(2, d): run_trade(DiscretePolicy(d), 2) for d in STEPS}
p4 = pd.DataFrame({k: row(v) for k, v in step_runs.items()}).T
p4.index.names = ["bps", "step"]
pd.concat([t1.loc[[(2, 0.0), (2, 0.1)]].rename(index=lambda b: b), p4])

sharpe  vol_ann  turnover_ann  cost_ann  max_drawdown  \
bps                                                                
2   0.000   1.329    0.086        16.562     0.003        -0.107   
    0.100   1.323    0.084        13.931     0.003        -0.106   
    0.003   1.329    0.086        16.560     0.003        -0.107   
    0.005   1.328    0.086        16.575     0.003        -0.108   
    0.010   1.323    0.086        16.602     0.003        -0.108   
    0.020   1.328    0.085        16.701     0.003        -0.106   

           tracking_risk  limit_override  
bps                                       
2   0.000          0.000           0.981  
    0.100          0.006           0.201  
    0.003          0.000           0.605  
    0.005          0.000           0.556  
    0.010          0.000           0.571  
    0.020          0.001           0.515

### Is any difference more than noise?

**Process.** Paired moving-block bootstrap of Sharpe(variant) minus Sharpe(10% band) at 2 bps (12-month blocks, 2,000 draws), then the deflated Sharpe of each difference series for the trials run.

In [6]:
from src.backtest.performance import deflated_sharpe

reference = band_runs[2, 0.10].net_return
variants = {f"band {b:.0%}": band_runs[2, b].net_return for b in BANDS if b != 0.10}
variants |= {f"step {d:.2%}": step_runs[2, d].net_return for d in STEPS}
diffs = {k: v - reference for k, v in variants.items()}
trial_sharpes = np.array([d.mean() / d.std(ddof=1) for d in diffs.values()])
rows = {}
for name, net in variants.items():
    rows[name] = {
        **paired_sharpe_difference(net, reference, block=12, draws=2000, seed=0),
        **{f"deflated_{n}": deflated_sharpe(diffs[name], trial_sharpes, n) for n in (len(diffs), 25, 100)},
    }
pd.DataFrame(rows).T

,difference,low_5pct,high_95pct,deflated_9,deflated_25,deflated_100
band 0%,0.006,-0.016,0.032,0.136,0.014,0.000
band 5%,0.002,-0.007,0.017,0.119,0.012,0.000
band 15%,-0.003,-0.019,0.006,0.000,0.000,0.000
band 20%,-0.009,-0.039,0.009,0.000,0.000,0.000
band 30%,-0.012,-0.071,0.025,0.000,0.000,0.000
step 0.25%,0.006,-0.016,0.033,0.138,0.014,0.000
step 0.50%,0.005,-0.018,0.033,0.075,0.006,0.000
step 1.00%,0.000,-0.026,0.031,0.021,0.001,0.000
step 2.00%,0.005,-0.031,0.042,0.002,0.000,0.000


- **The band cuts turnover by a sixth at almost no cost in Sharpe.** At 2 bps the 10% band takes turnover from 16.6x to 13.9x (-16%) and annual cost from 0.33% to 0.28% for 0.006 Sharpe (1.329 to 1.323). At 5 bps it is worth +0.003 Sharpe (1.271 to 1.274). Forecast tracking risk to the target is 0.6% a year, small against 8.4% portfolio volatility.
- **Wider bands cost Sharpe only at low cost levels.** The 20% band gives turnover 12.1x (-27%) and the 30% band 10.7x (-36%); at 2 bps they cost 0.009 and 0.012 Sharpe against the 10% band, at 5 bps nothing (1.271 and 1.272 against 1.274). Tracking risk rises to 1.2% and 1.7%.
- **With no band the executed limits fire in 98% of months:** the targets are scaled to sit exactly on the limits, so any fee pushes the book over and the limit cut applies. The 10% band lowers this to 20%.
- **Discretisation does nothing here.** Rounding to 0.25% to 2% steps leaves turnover at 16.6x to 16.7x, the same as no band, because the executed-limit step rescales every position each month and undoes the rounding. It is not a substitute for the band in this pipeline.
- **No difference is more than noise.** Every paired 90% range against the 10% band includes zero (no band: +0.006, -0.016 to 0.032; 30% band: -0.012, -0.071 to 0.025), and the deflated Sharpe of each difference is at most 0.14 for 9 trials and 0.014 at 25.
- **T2 is not testable** with a single cost rate for every market.

## Summary and recommendations

**What we found** (DCC-GARCH, HRP sizing, TREND, 2 bps and 5 bps, 1976-2014, in sample; 10 trade rules).
- The 10% band reduces turnover 16% and cost 15% for a Sharpe change inside the noise (-0.006 at 2 bps, +0.003 at 5 bps).
- Wider bands keep cutting turnover (-27% at 20%, -36% at 30%) and cost Sharpe only at 2 bps.
- Discretisation does not reduce turnover in this pipeline.
- Nothing is distinguishable from the 10% band after allowing for trials.

**Recommendations**
1. **Keep the 10% band.** It earns its place as a turnover control: it removes a sixth of the trading for no measurable loss.
2. **Consider 15% to 20% if real costs are 5 bps or more.** At 5 bps the Sharpe is flat from 0% to 30% while cost falls with turnover. Decide on a cost estimate, not on this backtest.
3. **Do not adopt size discretisation.** It adds a rule and no turnover saving here.
4. **Do not pursue a cost-based band (T2)** without per-market cost data; with one rate for every market there is nothing to scale the band by.
5. **Turnover is the weak point of the HRP default** (13.9x at 10% against 7.4x with inverse-volatility sizing). The bands help, but a cost-aware rule (`cost_aware.py`, from the earlier experiment) is the better test of cutting it.
6. **Single window, in sample.** Rerun on data not yet inspected before any adoption.